In [1]:
import cv2
import numpy as np
from pathlib import Path
from boxmot import BotSort
from ultralytics import YOLO

In [2]:
YOLO_MODEL_PATH = "./runs/detect/train/weights/best.pt"

In [3]:
model = YOLO(YOLO_MODEL_PATH)

In [4]:
tracker = BotSort(
    reid_weights=Path('osnet_x1_0_msmt17.pt'),
    device='0',
    half=False,
    with_reid=True,
)

2025-09-24 08:41:05.835 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.12 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
2025-09-24 08:41:05.837 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at osnet_x1_0_msmt17.pt; skipping download.
2025-09-24 08:41:06.090 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from osnet_x1_0_msmt17.pt


In [8]:
VIDEO_PATH = "video.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

In [9]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

In [10]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, stream=True)

    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy()

        detections = np.hstack((boxes, scores[:, np.newaxis], labels[:, np.newaxis]))

        if detections.size > 0:
            tracks = tracker.update(detections, frame)

            if tracks.size > 0:
                tracker.plot_results(frame, show_trajectories=False)



    cv2.imshow("YOLO + BoT-SORT Tracking", frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


0: 192x320 (no detections), 7.7ms
Speed: 1.1ms preprocess, 7.7ms inference, 2.1ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 9.4ms
Speed: 0.8ms preprocess, 9.4ms inference, 1.3ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 8.4ms
Speed: 0.8ms preprocess, 8.4ms inference, 1.5ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 9.3ms
Speed: 3.2ms preprocess, 9.3ms inference, 2.1ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 8.2ms
Speed: 0.9ms preprocess, 8.2ms inference, 0.7ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 14.7ms
Speed: 1.3ms preprocess, 14.7ms inference, 1.8ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 11.0ms
Speed: 1.1ms preprocess, 11.0ms inference, 1.2ms postprocess per image at shape (1, 3, 192, 320)

0: 192x320 (no detections), 9.8ms
Speed: 1.0ms preprocess, 9.8ms inference, 0.